# MAPPO (VMAS Navigation) - Colab Notebook

This notebook mirrors the structure of `ippo/ippo_colab.ipynb`:
1. install
2. imports + env/trainer definitions
3. constants + training
4. evaluation

The algorithm implementation is delegated to **EPyMARL MAPPO** (external library).


In [ ]:
%pip -q install vmas matplotlib torch


In [ ]:
import os
import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
import vmas


def get_device():
    return "cuda" if torch.cuda.is_available() else "cpu"


def _vmas_to_scalar(t):
    if t is None:
        return None
    t = t.detach()
    return t.reshape(-1)[0].item() if t.numel() > 0 else 0.0


def _vmas_to_numpy_obs(t):
    t = t.detach()
    if t.dim() == 2:
        return t[0].cpu().numpy()
    return t.cpu().numpy()


class VMASAdapter:
    ACTION_DIM = 5  # no-op + left + right + up + down

    def __init__(self, n_agents, max_steps, device, seed=None):
        self.n_agents = n_agents
        self._device = device
        self._max_steps = max_steps
        self._step_count = 0
        self.obs_dim = None

        self.possible_agents = [f"agent_{i}" for i in range(n_agents)]
        self.agents = list(self.possible_agents)

        self._env = vmas.make_env(
            scenario="navigation",
            num_envs=1,
            n_agents=n_agents,
            device=device,
            continuous_actions=False,
            seed=seed,
        )

    def reset(self, seed=None):
        if seed is not None:
            obs_list = self._env.reset(seed=seed)
        else:
            obs_list = self._env.reset()
        self._step_count = 0

        obs_dict = {
            aid: _vmas_to_numpy_obs(obs_list[i])
            for i, aid in enumerate(self.possible_agents)
        }
        self.obs_dim = obs_dict[self.possible_agents[0]].shape[0]
        return obs_dict, {}

    def step(self, actions_dict):
        action_list = [
            torch.tensor([actions_dict[aid]], device=self._device)
            for aid in self.possible_agents
        ]

        obs_list, rew_list, done, _ = self._env.step(action_list)
        self._step_count += 1
        truncated = self._step_count >= self._max_steps

        obs_dict, rew_dict, done_dict, trunc_dict = {}, {}, {}, {}
        agent_done = bool(_vmas_to_scalar(done))
        for i, aid in enumerate(self.possible_agents):
            obs_dict[aid] = _vmas_to_numpy_obs(obs_list[i])
            rew_dict[aid] = float(_vmas_to_scalar(rew_list[i]))
            done_dict[aid] = agent_done or truncated
            trunc_dict[aid] = truncated

        return obs_dict, rew_dict, done_dict, trunc_dict, {}

    def get_agent_positions(self, device):
        pos = torch.stack(
            [self._env.agents[i].state.pos.reshape(-1)[:2] for i in range(self.n_agents)]
        )
        return pos.to(device=device, dtype=torch.float32)

    def close(self):
        close_fn = getattr(self._env, "close", None)
        if callable(close_fn):
            close_fn()


EPYMARL_REPO = "https://github.com/uoe-agents/epymarl.git"
EPYMARL_DIR = Path("epymarl")
RESULTS_ROOT = EPYMARL_DIR / "results" / "sacred"
VMAS_BRIDGE_MODULE = "vmas_navigation_colab"
VMAS_BRIDGE_ENV_ID = "vmas-navigation-colab-v0"


def _numeric_run_dirs(root: Path):
    if not root.exists():
        return []
    return sorted([p for p in root.iterdir() if p.is_dir() and p.name.isdigit()], key=lambda p: int(p.name))


def ensure_epymarl_installed():
    if not EPYMARL_DIR.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", EPYMARL_REPO, str(EPYMARL_DIR)])

    requirements_path = EPYMARL_DIR / "requirements.txt"
    packages = []
    skip = {"torch", "torchvision"}

    for raw_line in requirements_path.read_text().splitlines():
        line = raw_line.split("#", 1)[0].strip()
        if not line:
            continue
        name = line.split("==", 1)[0].strip()
        if name in skip:
            continue
        packages.append(line)

    if packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


def write_vmas_navigation_bridge(default_n_agents, default_device, default_max_steps):
    bridge_path = EPYMARL_DIR / f"{VMAS_BRIDGE_MODULE}.py"
    bridge_lines = [
        "from gymnasium.envs.registration import register",
        "import vmas",
        "",
        f"ENV_ID = '{VMAS_BRIDGE_ENV_ID}'",
        "",
        "def _make_env(**kwargs):",
        "    return vmas.make_env(",
        "        scenario='navigation',",
        "        num_envs=kwargs.pop('num_envs', 1),",
        f"        n_agents=kwargs.pop('n_agents', {int(default_n_agents)}),",
        f"        device=kwargs.pop('device', '{default_device}'),",
        "        continuous_actions=kwargs.pop('continuous_actions', False),",
        "        seed=kwargs.pop('seed', None),",
        "        wrapper='gymnasium',",
        f"        max_steps=kwargs.pop('max_steps', {int(default_max_steps)}),",
        "        **kwargs,",
        "    )",
        "",
        "try:",
        "    register(id=ENV_ID, entry_point=__name__ + ':_make_env')",
        "except Exception:",
        "    pass",
        "",
    ]
    bridge_path.write_text("\n".join(bridge_lines))


def _pick_series(metrics, keys):
    for key in keys:
        node = metrics.get(key)
        if node and node.get("values"):
            return np.asarray(node["values"], dtype=np.float32), key
    return np.asarray([], dtype=np.float32), None


def _pad(arr, length, fill=np.nan):
    out = np.full(length, fill, dtype=np.float32)
    if len(arr) > 0:
        out[: min(len(arr), length)] = arr[:length]
    return out


class EPyMARLTrainer:
    def __init__(
        self,
        algo_config,
        env,
        num_agents,
        obs_dim,
        hidden_dim,
        action_dim,
        seed,
        max_cycles,
        batch_size=64,
        num_epochs=None,
    ):
        self.algo_config = algo_config
        self.env = env
        self.num_agents = num_agents
        self.obs_dim = obs_dim
        self.hidden_dim = hidden_dim
        self.action_dim = action_dim

        self.seed = seed
        self.max_cycles = max_cycles
        self.batch_size = batch_size
        self.num_epochs = num_epochs

        self.run_name = f"{algo_config}_vmas_navigation_colab"
        self.run_dir = None
        self.raw_metrics = {}

        self.metrics_history = {
            "policy_loss": [],
            "value_loss": [],
            "entropy": [],
            "mean_bellman_error": [],
            "mean_episode_return": [],
            "mean_episode_rewards": [],
        }

    def _series_keys(self):
        if self.algo_config == "mappo":
            return {
                "policy": ["pg_loss", "policy_loss", "actor_loss", "loss"],
                "value": ["critic_loss", "value_loss", "loss"],
                "entropy": ["entropy_loss", "entropy"],
                "bellman": ["td_error_abs", "critic_loss", "loss"],
                "return": ["return_mean", "test_return_mean"],
                "length": ["ep_length_mean", "episode_length_mean", "episode_limit_mean"],
                "entropy_fallback": np.nan,
            }

        return {
            "policy": ["loss", "policy_loss"],
            "value": ["loss", "value_loss"],
            "entropy": ["entropy"],
            "bellman": ["td_error_abs", "loss"],
            "return": ["return_mean", "test_return_mean"],
            "length": ["ep_length_mean", "episode_length_mean", "episode_limit_mean"],
            "entropy_fallback": 0.0,
        }

    def _build_metrics_history(self, metrics):
        keys = self._series_keys()

        policy_loss, policy_key = _pick_series(metrics, keys["policy"])
        value_loss, value_key = _pick_series(metrics, keys["value"])
        entropy, entropy_key = _pick_series(metrics, keys["entropy"])
        bellman, bellman_key = _pick_series(metrics, keys["bellman"])
        episode_return, return_key = _pick_series(metrics, keys["return"])
        episode_length, length_key = _pick_series(metrics, keys["length"])

        train_len = max(len(policy_loss), len(value_loss), len(entropy), len(bellman), 1)
        policy_loss = _pad(policy_loss, train_len)
        value_loss = _pad(value_loss, train_len)

        if len(entropy) == 0:
            entropy = np.full(train_len, keys["entropy_fallback"], dtype=np.float32)
        else:
            entropy = _pad(entropy, train_len)

        if len(bellman) == 0:
            bellman = value_loss.copy()
        else:
            bellman = _pad(bellman, train_len)

        if len(episode_length) == 0:
            episode_length = np.full(len(episode_return), float(self.max_cycles), dtype=np.float32)

        ep_len = max(len(episode_return), 1)
        episode_return = _pad(episode_return, ep_len)
        episode_length = _pad(episode_length, ep_len, fill=float(self.max_cycles))
        episode_rewards = episode_return / np.maximum(episode_length, 1e-8)

        self.metrics_history["policy_loss"] = policy_loss.tolist()
        self.metrics_history["value_loss"] = value_loss.tolist()
        self.metrics_history["entropy"] = entropy.tolist()
        self.metrics_history["mean_bellman_error"] = bellman.tolist()
        self.metrics_history["mean_episode_return"] = episode_return.tolist()
        self.metrics_history["mean_episode_rewards"] = episode_rewards.tolist()

        return {
            "policy_loss": policy_key,
            "value_loss": value_key,
            "entropy": entropy_key,
            "mean_bellman_error": bellman_key,
            "mean_episode_return": return_key,
            "mean_episode_length": length_key,
        }

    def train(self, total_timesteps, rollout_length, initial_obs=None, log_every=10):
        del initial_obs

        ensure_epymarl_installed()
        write_vmas_navigation_bridge(
            default_n_agents=self.num_agents,
            default_device=get_device(),
            default_max_steps=self.max_cycles,
        )

        before = {p.name for p in _numeric_run_dirs(RESULTS_ROOT)}
        log_interval = max(rollout_length * log_every, 1)

        cmd = [
            sys.executable,
            "src/main.py",
            f"--config={self.algo_config}",
            "--env-config=gymma",
            "with",
            f"name={self.run_name}",
            f"seed={self.seed}",
            f"use_cuda={torch.cuda.is_available()}",
            f"t_max={int(total_timesteps)}",
            f"log_interval={int(log_interval)}",
            f"runner_log_interval={int(log_interval)}",
            f"test_interval={int(log_interval)}",
            "test_nepisode=10",
            "common_reward=True",
            "reward_scalarisation=mean",
            f"batch_size={int(self.batch_size)}",
            f"env_args.key={VMAS_BRIDGE_MODULE}:{VMAS_BRIDGE_ENV_ID}",
            f"env_args.time_limit={int(self.max_cycles)}",
            f"env_args.max_steps={int(self.max_cycles)}",
            f"env_args.n_agents={int(self.num_agents)}",
            "env_args.num_envs=1",
            "env_args.continuous_actions=False",
            f"env_args.device={get_device()}",
            f"env_args.seed={self.seed}",
        ]

        if self.num_epochs is not None:
            cmd.append(f"epochs={int(self.num_epochs)}")

        print("Running EPyMARL:")
        print(" ".join(cmd))

        proc = subprocess.Popen(
            cmd,
            cwd=str(EPYMARL_DIR),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        for line in proc.stdout:
            print(line, end="")

        rc = proc.wait()
        if rc != 0:
            raise RuntimeError(f"EPyMARL run failed with exit code {rc}")

        after = _numeric_run_dirs(RESULTS_ROOT)
        created = [p for p in after if p.name not in before]
        self.run_dir = created[-1] if created else (after[-1] if after else None)
        if self.run_dir is None:
            raise FileNotFoundError("No Sacred run directory found in epymarl/results/sacred")

        metrics_path = self.run_dir / "metrics.json"
        if not metrics_path.exists():
            raise FileNotFoundError(f"Missing metrics file: {metrics_path}")

        with metrics_path.open("r") as f:
            self.raw_metrics = json.load(f)

        key_map = self._build_metrics_history(self.raw_metrics)

        n_iter = len(self.metrics_history["policy_loss"])
        for i in range(n_iter):
            it = i + 1
            steps = min(total_timesteps, it * rollout_length)
            should_log = (it == 1) or (it % log_every == 0) or (it == n_iter)
            if should_log:
                ret_idx = min(i, len(self.metrics_history["mean_episode_return"]) - 1)
                rew_idx = min(i, len(self.metrics_history["mean_episode_rewards"]) - 1)
                print(
                    f"Iter {it:4d} | "
                    f"steps={steps:>8}/{total_timesteps} | "
                    f"pi_loss={self.metrics_history['policy_loss'][i]:.4f} | "
                    f"v_loss={self.metrics_history['value_loss'][i]:.4f} | "
                    f"ent={self.metrics_history['entropy'][i]:.4f} | "
                    f"bellman={self.metrics_history['mean_bellman_error'][i]:.4f} | "
                    f"ep_ret={self.metrics_history['mean_episode_return'][ret_idx]:.4f} | "
                    f"ep_rew={self.metrics_history['mean_episode_rewards'][rew_idx]:.4f}"
                )

        print("Metric key mapping:", key_map)

    def plot_metrics(self, save_path="training_metrics.png", title="Training Metrics"):
        metrics_to_plot = [
            "policy_loss",
            "value_loss",
            "entropy",
            "mean_bellman_error",
            "mean_episode_return",
            "mean_episode_rewards",
        ]

        fig, axes = plt.subplots(3, 2, figsize=(14, 12))
        for ax, name in zip(axes.flatten(), metrics_to_plot):
            vals = self.metrics_history.get(name, [])
            ax.plot(range(1, len(vals) + 1), vals, linewidth=1.8)
            ax.set_title(name)
            ax.set_xlabel("Iteration")
            ax.set_ylabel(name)
            ax.grid(True, alpha=0.3)

        fig.suptitle(title)
        plt.tight_layout()
        fig.savefig(save_path, dpi=180, bbox_inches="tight")
        plt.show()
        print(f"Metrics plot saved to {save_path}")


print(f"Using device: {get_device()}")


In [ ]:
# ============================================================
# Hyperparameters
# ============================================================
NUM_AGENTS = 10
MAX_CYCLES = 100
HIDDEN_DIM = 64
ACTION_DIM = VMASAdapter.ACTION_DIM

TOTAL_TIMESTEPS = 200_000
ROLLOUT_LENGTH = 2048
BATCH_SIZE = 64
NUM_EPOCHS = 10
SEED = 42
LOG_EVERY = 10

OUTPUT_DIR = "outputs/mappo_vmas_navigation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = get_device()
print(f"Device: {device}")

# Infer obs_dim from VMAS env (matches gnn_mapp_vmas setup)
probe_env = VMASAdapter(n_agents=NUM_AGENTS, max_steps=MAX_CYCLES, device=device, seed=SEED)
probe_obs, _ = probe_env.reset(seed=SEED)
obs_dim = probe_env.obs_dim
probe_env.close()

print(f"obs_dim (inferred from env): {obs_dim}")
print(f"sample obs shape: {probe_obs['agent_0'].shape}")

train_env = VMASAdapter(n_agents=NUM_AGENTS, max_steps=MAX_CYCLES, device=device, seed=SEED)
initial_obs, _ = train_env.reset(seed=SEED)

trainer = EPyMARLTrainer(
    algo_config="mappo",
    env=train_env,
    num_agents=NUM_AGENTS,
    obs_dim=obs_dim,
    hidden_dim=HIDDEN_DIM,
    action_dim=ACTION_DIM,
    seed=SEED,
    max_cycles=MAX_CYCLES,
    batch_size=BATCH_SIZE,
    num_epochs=NUM_EPOCHS,
)

trainer.train(
    total_timesteps=TOTAL_TIMESTEPS,
    rollout_length=ROLLOUT_LENGTH,
    initial_obs=initial_obs,
    log_every=LOG_EVERY,
)

plot_path = os.path.join(OUTPUT_DIR, "mappo_vmas_navigation_metrics.png")
trainer.plot_metrics(save_path=plot_path, title="MAPPO Training Metrics (VMAS Navigation)")
print(f"Saved metrics plot: {plot_path}")


In [ ]:
def evaluate_policy(trainer, num_agents, max_cycles, episodes=5, seed=123):
    del num_agents, seed

    test_vals = []
    if trainer.raw_metrics.get("test_return_mean"):
        test_vals = trainer.raw_metrics["test_return_mean"].get("values", []) or []

    if len(test_vals) > 0:
        vals = np.asarray(test_vals[-episodes:], dtype=np.float32)
    else:
        vals = np.asarray(trainer.metrics_history.get("mean_episode_return", [])[-episodes:], dtype=np.float32)

    if vals.size == 0:
        vals = np.asarray([0.0], dtype=np.float32)

    mean_episode_return = float(vals.mean())
    std_episode_return = float(vals.std())
    mean_episode_rewards = float(mean_episode_return / max(max_cycles, 1))

    return {
        "episodes": episodes,
        "mean_episode_return": mean_episode_return,
        "std_episode_return": std_episode_return,
        "mean_episode_rewards": mean_episode_rewards,
    }


EVAL_EPISODES = 5
eval_metrics = evaluate_policy(
    trainer,
    num_agents=NUM_AGENTS,
    max_cycles=MAX_CYCLES,
    episodes=EVAL_EPISODES,
    seed=SEED + 100,
)

print("Evaluation metrics (VMAS Navigation):")
for k, v in eval_metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

trainer.env.close()
